# Bayesian Network Inference and Tracking

Implementation of Bayesian-network construction, factor operations, enumeration, variable elimination, and probabilistic tracking logic.

**Topics:** Bayesian networks, Factor operations, Inference by enumeration, Variable elimination, Hidden-state tracking

> Clean portfolio version: official problem statements, grading instructions, and personal/student identifiers were removed. The remaining code represents my implementation and learning outcomes from an undergraduate Artificial Intelligence course.


In [ ]:
import cv2
from google.colab.patches import cv2_imshow
from google.colab import output
import time
import pygame
import os, sys

In [ ]:
os.environ["SDL_VIDEODRIVER"] = "dummy"
pygame.display.init()
pygame.display.set_mode((1, 1), pygame.NOFRAME)
pygame.font.init()

In [ ]:
!pip install gdown
!gdown 1qX5euDVHAg4PDo9TnyUuBp9o2IK9Hjod
!unzip -o src.zip

In [ ]:
import automobile
import bayes_net as bn
import functools
from copy import deepcopy

from game import GameState, AgentState
from typing import List, Dict, Optional

from inference import DiscreteDistribution
from util import manhattan_distance
from automobile import Agent
from distance_calculator import Distancer
from game import Actions

In [ ]:
def draw(state: GameState, agents):
    from display import draw as d
    view = d(state, agents)
    img_bgr = cv2.cvtColor(view, cv2.COLOR_RGB2BGR)
    output.clear(wait=True)
    cv2_imshow(img_bgr)
    import time
    time.sleep(0.1)

import tests.game_utils
tests.game_utils.draw_function = draw

# **ماشین بستنی فروشی خودران**


# **بخش اول: ساختار بیز نت**


In [ ]:
def construct_bayes_net(game_state: GameState) -> bn.BayesNet:
    """
    Construct an empty Bayes net according to the structure given.

    You *must* name all variables using the constants in this function.

    In this method, you should:
    - populate `variables` with the Bayes Net nodes
    - populate `edges` with every edge in the Bayes Net. we will represent each
      edge as a tuple `(from, to)`.
    - set each `variable_domains_dict[var] = values`, where `values` is a list
      of the possible assignments to `var`.
        - each agent position is a tuple (x, y) where x and y are 0-indexed
        - each observed distance is a noisy Manhattan distance:
          it's non-negative and |obs - true| <= MAX_NOISE
    - this uses slightly simplified mechanics vs the ones used later for simplicity
    """
    # constants to use
    CAR = "Car"
    FINN = "Finn"
    JAKE = "Jake"
    OBS0 = "Observation0"
    OBS1 = "Observation1"
    X_RANGE = game_state.get_walls().width
    Y_RANGE = game_state.get_walls().height
    MAX_NOISE = 7

    variables = []
    edges = []
    variable_domains_dict = {}

    # *** YOUR CODE HERE ***
    
    # *** END YOUR CODE HERE ***

    net = bn.construct_empty_bayes_net(variables, edges, variable_domains_dict)
    return net

# تست‌های بخش اول


In [ ]:
from tests import q1

q1.test_1(construct_bayes_net)
q1.test_2(construct_bayes_net)

# **بخش دوم: جوین کردن فاکتورهای داده شده**


In [ ]:
def join_factors(factors: List[bn.Factor]) -> bn.Factor:
    """
    Input factors is a list of factors.

    You should calculate the set of unconditioned variables and conditioned
    variables for the join of those factors.

    Return a new factor that has those variables and whose probability entries
    are product of the corresponding rows of the input factors.

    You may assume that the variable_domains_dict for all the input
    factors are the same, since they come from the same BayesNet.

    joinFactors will only allow unconditioned_variables to appear in
    one input factor (so their join is well defined).

    Hint: Factor methods that take an assignmentDict as input
    (such as get_probability and set_probability) can handle
    assignmentDicts that assign more variables than are in that factor.

    Useful functions:
    Factor.get_all_possible_assignment_dicts
    Factor.get_probability
    Factor.set_probability
    Factor.unconditioned_variables
    Factor.conditioned_variables
    Factor.variable_domains_dict
    """

    factors = list(factors)

    # Ignore this part
    sets_of_unconditioned = [set(factor.unconditioned_variables()) for factor in factors]
    if len(factors) > 1:
        intersect = functools.reduce(lambda x, y: x & y, sets_of_unconditioned)
        if len(intersect) > 0:
            print("Factor failed joinFactors typecheck: ", factors)
            raise ValueError(
                "unconditioned_variables can only appear in one factor. \n"
                + "unconditioned_variables: " + str(intersect) +
                "\nappear in more than one input factor.\n" +
                "Input factors: \n" +
                "\n".join(map(str, factors))
            )

    # Find out which variables are unconditioned and conditioned in the final joined factor

    # *** YOUR CODE HERE ***
    
    # *** END YOUR CODE HERE ***

    joined_factor = bn.Factor(
        final_unconditioned_variables,
        final_conditioned_variables,
        factors[0].variable_domains_dict()
    )

    # Use Factor.set_probability and find probabilities for each assignment of variables
    # Hint: use Factor.get_all_possible_assignment_dicts

    # *** YOUR CODE HERE ***
 
    # *** END YOUR CODE HERE ***

    return joined_factor

# تست‌های بخش دوم


In [ ]:
from tests import q2

q2.test_product_rule(join_factors)
q2.test_product_rule_extended(join_factors)
q2.test_disjoint_right(join_factors)
q2.test_common_right(join_factors)
q2.test_grade_join(join_factors)
q2.test_product_rule_nonsingleton_var(join_factors)

# **بخش سوم: Eliminate**


In [ ]:
def eliminate(factor: bn.Factor, elimination_variable: str) -> bn.Factor:
    """
    Input factor is a single factor.
    Input elimination_variable is the variable to eliminate from factor.
    elimination_variable must be an unconditioned variable in factor.

    You should calculate the set of unconditioned variables and conditioned
    variables for the factor obtained by eliminating the variable
    elimination_variable.

    Return a new factor where all of the rows mentioning
    elimination_variable are summed with rows that match
    assignments on the other variables.

    Useful functions:
    Factor.get_all_possible_assignment_dicts
    Factor.get_probability
    Factor.set_probability
    Factor.unconditioned_variables
    Factor.conditioned_variables
    Factor.variable_domains_dict
    """

    # Ignore this part
    if elimination_variable not in factor.unconditioned_variables():
        print("Factor failed eliminate typecheck: ", factor)
        raise ValueError(
            "Elimination variable is not an unconditioned variable " \
            + "in this factor\n" +
            "elimination_variable: " + str(elimination_variable) + \
            "\nunconditioned_variables:" + str(factor.unconditioned_variables())
        )

    if len(factor.unconditioned_variables()) == 1:
        print("Factor failed eliminate typecheck: ", factor)
        raise ValueError(
            "Factor has only one unconditioned variable, so you " \
            + "can't eliminate \nthat variable.\n" + \
            "elimination_variable:" + str(elimination_variable) + "\n" + \
            "unconditioned_variables: " + str(factor.unconditioned_variables())
        )

    # *** YOUR CODE HERE ***"
    
    # *** END YOUR CODE HERE ***"

# تست‌های بخش سوم


In [ ]:
from tests import q3

q3.test_simple_eliminate(eliminate)
q3.test_simple_eliminate_extended(eliminate)
q3.test_eliminate_conditioned(eliminate)
q3.test_grade_eliminate(eliminate)
q3.test_simple_eliminate_nonsingleton_var(eliminate)
q3.test_simple_eliminate_int(eliminate)

# **بخش چهارم: Enumeration**


In [ ]:
def join_factors_by_variable(factors: List[bn.Factor], join_variable: str):
    """
    Input factors is a list of factors.
    Input joinVariable is the variable to join on.

    This function performs a check that the variable that is being joined on
    appears as an unconditioned variable in only one of the input factors.

    Then, it calls your joinFactors on all of the factors in factors that
    contain that variable.

    Returns a tuple of
    (factors not joined, resulting factor from joinFactors)
    """

    factors_to_join = [factor for factor in factors if join_variable in factor.variables_set()]
    factors_not_to_join = [factor for factor in factors if join_variable not in factor.variables_set()]

    # Ignore this part
    num_variables_on_left = len(
        [factor for factor in factors_to_join if join_variable in factor.unconditioned_variables()]
    )
    if num_variables_on_left > 1:
        print("Factor failed joinFactorsByVariable typecheck: ", factors)
        raise ValueError(
            "The joinBy variable can only appear in one factor as an \nunconditioned variable. \n" +
            "join_variable: " + str(join_variable) + "\n" +
            ", ".join(map(str, [factor.unconditioned_variables() for factor in factors_to_join]))
        )

    joined_factor = join_factors(factors_to_join)
    return factors_not_to_join, joined_factor

In [ ]:
def inference_by_enumeration(bayes_net: bn.BayesNet, query_variables: List[str], evidence_dict: Dict):
    """
    An inference by enumeration implementation provided as reference.
    This function performs a probabilistic inference query that
    returns the factor:

    P(query_variables | evidence_dict)

    bayes_net:       The Bayes Net on which we are making a query.
    query_variables: A list of the variables which are unconditioned in
                    the inference query.
    evidence_dict:   An assignment dict {variable : value} for the
                    variables which are presented as evidence
                    (conditioned) in the inference query. 
    """

    # initialize return variables and the variables to eliminate
    evidence_variables_set = set(evidence_dict.keys())
    query_variables_set = set(query_variables)
    elimination_variables = (bayes_net.variables_set() - evidence_variables_set) - query_variables_set

    # grab all factors where we know the evidence variables (to reduce the size of the tables)
    current_factors_list = bayes_net.get_all_cpts_with_evidence(evidence_dict)

    # join all factors by variables
    # *** YOUR CODE HERE ***
    
    # *** END YOUR CODE HERE ***

    # current_factors_list should contain the connected components of the graph now as factors, must join the connected components
    full_join = join_factors(current_factors_list)

    # marginalize all variables that aren't query or evidence
    incrementally_marginalized_joint = full_join
    # *** YOUR CODE HERE ***

    # *** END YOUR CODE HERE ***

    full_joint_over_query_and_evidence = incrementally_marginalized_joint

    # normalize so that the probability sums to one
    # the input factor contains only the query variables and the evidence variables, 
    # both as unconditioned variables
    query_conditioned_on_evidence = bn.normalize(full_joint_over_query_and_evidence)
    # now the factor is conditioned on the evidence variables

    return query_conditioned_on_evidence

# تست‌های بخش چهارم


In [ ]:
from tests import q4

q4.test_disconnected_eliminate(inference_by_enumeration)
q4.test_independent_eliminate(inference_by_enumeration)
q4.test_independent_eliminate_extended(inference_by_enumeration)
q4.test_common_effect_eliminate(inference_by_enumeration)
q4.test_grade_var_elim(inference_by_enumeration)
q4.test_large_bayes_net_elim(inference_by_enumeration)

# **بخش پنجم: Variable Elimination**


In [ ]:
def inference_by_variable_elimination(
        bayes_net: bn.BayesNet,
        query_variables: List[str],
        evidence_dict: Dict,
        elimination_order: List[str]
):
    """
    This function should perform a probabilistic inference query that
    returns the factor:

    P(query_variables | evidence_dict)

    It should perform inference by interleaving joining on a variable
    and eliminating that variable, in the order of variables according
    to elimination_order.

    You need to use joinFactorsByVariable to join all of the factors
    that contain a variable in order for the autograder to
    recognize that you performed the correct interleaving of
    joins and eliminates.

    If a factor that you are about to eliminate a variable from has
    only one unconditioned variable, you should not eliminate it
    and instead just discard the factor.  This is since the
    result of the eliminate would be 1 (you marginalize
    all of the unconditioned variables), but it is not a
    valid factor.  So this simplifies using the result of eliminate.

    The sum of the probabilities should sum to one (so that it is a true
    conditional probability, conditioned on the evidence).

    bayes_net:         The Bayes Net on which we are making a query.
    query_variables:   A list of the variables which are unconditioned
                      in the inference query.
    evidence_dict:     An assignment dict {variable : value} for the
                      variables which are presented as evidence
                      (conditioned) in the inference query.
    elimination_order: The order to eliminate the variables in.

    Hint: BayesNet.get_all_cpts_with_evidence will return all the Conditional
    Probability Tables even if an empty dict (or None) is passed in for
    evidence_dict. In this case it will not specialize any variable domains
    in the CPTs.

    Useful functions:
    BayesNet.get_all_cpts_with_evidence
    normalize
    eliminate
    join_factors_by_variable
    join_factors
    """

    if elimination_order is None:  # set an arbitrary elimination order if None given
        elimination_variables = bayes_net.variables_set() - set(query_variables) - \
                                set(evidence_dict.keys())
        elimination_order = sorted(list(elimination_variables))

    # *** YOUR CODE HERE ***
  
    # *** END YOUR CODE HERE ***

# تست‌های بخش پنجم


In [ ]:
from tests import q5

q5.test_disconnected_eliminate(inference_by_variable_elimination)
q5.test_independent_eliminate(inference_by_variable_elimination)
q5.test_independent_eliminate_extended(inference_by_variable_elimination)
q5.test_common_effect_eliminate(inference_by_variable_elimination)
q5.test_grade_var_elim(inference_by_variable_elimination)
q5.test_large_bayes_net_elim(inference_by_variable_elimination)

# **بخش ششم: Observation Probability**


In [ ]:
class InferenceModule:
    """
    An inference module tracks a belief distribution over a person's location.
    """

    def __init__(self, person_agent):
        """
        set the person agent for later access.
        """
        self.person_agent = person_agent
        self.index = person_agent.index
        self.obs = []  # most recent observation position

    def get_home_position(self):
        return 2 * self.index - 1, 1

    def get_position_distribution_helper(self, game_state, pos, index, agent):
        try:
            home = self.get_home_position()
            game_state = self.set_person_position(game_state, pos, index)
        except TypeError:
            home = self.get_home_position(index)
            game_state = self.set_person_positions(game_state, pos)
        car_position = game_state.get_car_position()
        person_position = game_state.get_person_position(index)  # the position you set
        dist = DiscreteDistribution()
        if car_position == person_position:  # the person has been caught!
            dist[home] = 1.0
            return dist
        car_successor_states = automobile.Actions.get_legal_neighbors(
            car_position,
            game_state.get_walls()
        )  # positions car can move to
        if person_position in car_successor_states:  # person could get caught
            mult = 1.0 / float(len(car_successor_states))
            dist[home] = mult
        else:
            mult = 0.0
        action_dist = agent.get_distribution(game_state)
        for action, prob in action_dist.items():
            successor_position = automobile.Actions.get_successor(person_position, action)
            if successor_position in car_successor_states:  # person could get caught
                denom = float(len(action_dist))
                dist[home] += prob * (1.0 / denom) * (1.0 - mult)
                dist[successor_position] = prob * ((denom - 1.0) / denom) * (1.0 - mult)
            else:
                dist[successor_position] = prob * (1.0 - mult)
        return dist

    def get_position_distribution(self, game_state, pos, index=None, agent=None):
        """
        return a distribution over successor positions of the person from the
        given game_state. you must first place the person in the game_state, using
        set_person_position below.
        """
        if index == None:
            index = self.index
        if agent == None:
            agent = self.person_agent
        return self.get_position_distribution_helper(game_state, pos, index, agent)

    def get_observation_prob(
            self,
            noisy_distance: int,
            car_position: tuple,
            person_position: tuple,
            home_position: tuple
    ):
        """
        return the probability p(noisy_distance | car_position, person_position).
        """
        # *** YOUR CODE HERE ***
        
        # *** END YOUR CODE HERE ***

    def set_person_position(self, game_state, person_position, index):
        game_state = deepcopy(game_state)
        game_state.set_person_state(index, AgentState(person_position, False, automobile.Directions.stop))
        return game_state

    def set_person_positions(self, game_state, person_positions):
        """
        sets the position of all persons to the values in person_positions.
        """
        game_state = deepcopy(game_state)
        for index, pos in enumerate(person_positions):
            from game import AgentState
            game_state.set_person_state(index + 1, AgentState(pos, False, automobile.Directions.stop))
        return game_state

    def observe(self, game_state):
        """
        collect the relevant noisy distance observation and pass it along.
        """
        distances = game_state.get_noisy_person_distances()
        if len(distances) >= self.index:  # check for missing observations
            obs = distances[self.index - 1]
            self.obs = obs
            self.observe_update(obs, game_state)

    def initialize(self, game_state):
        """
        initialize beliefs to a uniform distribution over all legal positions.
        """
        self.legal_positions = [p for p in game_state.get_walls().as_list(False) if p[1] > 1]
        self.all_positions = self.legal_positions + [self.get_home_position()]
        self.initialize_uniformly(game_state)

    def initialize_uniformly(self, game_state):
        """
        set the belief state to a uniform prior belief over all positions.
        """
        raise NotImplementedError

    def observe_update(self, observation: Optional[int], game_state):
        """
        update beliefs based on the given distance observation and game_state.
        """
        raise NotImplementedError

    def elapse_time(self, game_state):
        """
        predict beliefs for the next time step from a game_state.
        """
        raise NotImplementedError

    def get_belief_distribution(self):
        """
        return the agent's current belief state, a distribution over person
        locations conditioned on all evidence so far.
        """
        raise NotImplementedError

# تست‌های بخش ششم


In [ ]:
from tests import q6

q6.test(InferenceModule)

# **بخش هفتم: Exact Inference Observation**


In [ ]:
class ExactInference(InferenceModule):
    """
    The exact dynamic inference module should use forward algorithm updates to
    compute the exact belief function at each time step.
    """

    def initialize_uniformly(self, game_state):
        """
        Begin with a uniform distribution over legal person positions (i.e., not
        including the home position).
        """
        self.beliefs = DiscreteDistribution()
        game_state: GameState
        for p in self.legal_positions:
            self.beliefs[p] = 1.0
        self.beliefs.normalize()

    def observe_update(self, observation: int, game_state: GameState):
        """
        Update beliefs based on the distance observation and car's position.

        The observation is the noisy Manhattan distance to the person you are
        tracking.

        self.all_positions is a list of the possible person positions, including
        the home position. You should only consider positions that are in
        self.all_positions.

        The update model is not entirely stationary: it may depend on car's
        current position. However, this is not a problem, as car's current
        position is known.
        """
        # "*** YOUR CODE HERE ***"

        # "*** END YOUR CODE HERE ***"
        self.beliefs.normalize()

    def elapse_time(self, game_state: GameState):
        """
        Predict beliefs in response to a time step passing from the current
        state.

        The transition model is not entirely stationary: it may depend on
        car's current position. However, this is not a problem, as car's
        current position is known.
        """
        # "*** YOUR CODE HERE ***"
      
        # "*** END YOUR CODE HERE ***"

    def get_belief_distribution(self):
        return self.beliefs

# تست‌های بخش هفتم


In [ ]:
from tests import q7
q7.test_5(ExactInference, draw=True)

In [ ]:
from tests import q7

q7.test_1(ExactInference)
q7.test_2(ExactInference)
q7.test_3(ExactInference)
q7.test_4(ExactInference)
q7.test_5(ExactInference)
q7.test_6(ExactInference)
q7.test_7(ExactInference)
q7.test_8(ExactInference)

# **بخش هشتم: Greedy Agent**


In [ ]:
class GreedyCarAgent(Agent):
    """An agent that charges the closest person."""

    def __init__(
            self,
            index,
            inference_class=ExactInference,
            person_agents=None,
            observe_enable=True,
            elapse_time_enable=True
    ):
        super().__init__(index)
        self.inferences = [inference_class(agent) for agent in person_agents]
        self.observe_enable = observe_enable
        self.elapse_time_enable = elapse_time_enable
        self.distancer = None
        self.first_move = True

    def registerInitialState(self, game_state: GameState):
        """Pre-computes the distance between every two points."""
        for inference in self.inferences:
            inference.initialize(game_state)
        self.distancer = Distancer(game_state.layout)

    def get_action(self, game_state: GameState):
        """Updates beliefs, then chooses an action based on updated beliefs."""
        for index, inf in enumerate(self.inferences):
            if not self.first_move and self.elapse_time_enable:
                inf.elapse_time(game_state)
            self.first_move = False
            if self.observe_enable:
                inf.observe(game_state)
        return self.choose_action(game_state)

    def choose_action(self, game_state: GameState):
        """
        First computes the most likely position of each person that has
        not yet been captured, then chooses an action that brings
        the car closest to the closest person (according to mazeDistance!).

        Hint: Use the following variables in your code.
        """
        car_position = game_state.get_car_position()
        legal_actions = [a for a in game_state.get_legal_actions(0)]
        not_eaten_people = game_state.get_not_eaten_people()
        not_eaten_people_position_distribution = [inference.get_belief_distribution() for i, inference in
                                                  enumerate(self.inferences)
                                                  if not_eaten_people[inference.index - 1]]
        # "*** YOUR CODE HERE ***"

        # "*** END YOUR CODE HERE ***"

# تست‌های بخش هشتم


In [ ]:
from tests import q8

q8.test_1(GreedyCarAgent, ExactInference, draw=True)

In [ ]:
q8.test_1(GreedyCarAgent, ExactInference)

# **بخش نهم: Particle Filter**


In [ ]:
state = None
class ParticleFilter(InferenceModule):
    """
    A particle filter for approximately tracking a single person.
    """

    def __init__(self, person_agent, num_particles=5000):
        super().__init__(person_agent)
        self.set_num_particles(num_particles)

    def set_num_particles(self, num_particles):
        self.num_particles = num_particles

    def initialize_uniformly(self, game_state: GameState):
        """
        Initialize a list of particles. Use self.num_particles for the number of
        particles. Use self.legal_positions for the legal board positions where
        a particle could be located. Particles should be evenly (not randomly)
        distributed across positions in order to ensure a uniform prior. Use
        self.particles for the list of particles.
        """
        self.particles = []
        # "*** YOUR CODE HERE ***"

        # "*** END YOUR CODE HERE ***"

    def get_belief_distribution(self):
        """
        Return the agent's current belief state, a distribution over person
        locations conditioned on all evidence and time passage. This method
        essentially converts a list of particles into a belief distribution.

        This function should return a normalized distribution.
        """
        # "*** YOUR CODE HERE ***"

        # "*** END YOUR CODE HERE ***"

    def observe_update(self, observation: int, game_state: GameState):
        """
        Update beliefs based on the distance observation and car's position.

        The observation is the noisy Manhattan distance to the person you are
        tracking.

        There is one special case that a correct implementation must handle.
        When all particles receive zero weight, the list of particles should
        be reinitialized by calling initializeUniformly. The total method of
        the DiscreteDistribution may be useful.
        """
        # "*** YOUR CODE HERE ***"
       
        # "*** END YOUR CODE HERE ***"

    def elapse_time(self, game_state):
        """
        Sample each particle's next state based on its current state and the
        gameState.
        """
        # "*** YOUR CODE HERE ***"

        # "*** END YOUR CODE HERE ***"

# تست‌های بخش نهم


In [ ]:
from tests import q9
q9.test_7(GreedyCarAgent, ParticleFilter, draw=True)

In [ ]:
from tests import q9

q9.test_1(ParticleFilter)
q9.test_2(ParticleFilter)
q9.test_3(ParticleFilter)
q9.test_4(ParticleFilter)
q9.test_5(ParticleFilter)
q9.test_6(ParticleFilter)
q9.test_7(GreedyCarAgent, ParticleFilter)